In [0]:
dbutils.widgets.text("url", "https://realtime.hsl.fi/realtime/trip-updates/v2/hsl")
dbutils.widgets.text("message_type", "trip_updates")

url = dbutils.widgets.get("url")
message_type = dbutils.widgets.get("message_type")

In [0]:
from pyspark.sql.protobuf.functions import from_protobuf

descriptor_path = "/Volumes/shared/john_armstrong/gtfs_rt/gtfs_rt.desc"
message_name = "transit_realtime.FeedMessage"  # Full package name from gtfs-realtime.proto

In [0]:
import requests
import threading
from pyspark.sql.datasource import SimpleDataSourceStreamReader, DataSourceStreamReader, DataSource, InputPartition
from pyspark.sql.types import StructType
from typing import Iterator, Tuple
import time

from pyspark.sql.types import (
    StructType, StructField, BinaryType, StringType, IntegerType
)
from pyspark.sql.functions import current_timestamp
from typing import Iterator, Tuple
import time
from datetime import datetime, timezone

class GTFSRTSimpleStreamReader(SimpleDataSourceStreamReader):
    def __init__(self, schema: StructType, options: dict):
        self.schema = schema
        self.url = options["url"]
        self.session = requests.Session()
        self.buffer = []  # List of (offset, url, content)
        self.current_offset = 0
        self.thread = threading.Thread(target=self._fetch_loop, daemon=True)
        self.thread.start()

    def _fetch_loop(self):
        while True:
            try:
                resp = self.session.get(self.url)
                resp.raise_for_status()
                self.buffer.append((self.current_offset, self.url, resp.content))
                self.current_offset += 1
            except Exception as e:
                print(f"Error fetching data: {e}")
            time.sleep(1)

    def initialOffset(self) -> dict:
        return {"offset": 0}

    def read(self, start: dict) -> Tuple[Iterator[Tuple[int, str, bytes]], dict]:
        start_offset = start["offset"]
        # Remove data already processed (offsets < start_offset)
        self.buffer = [item for item in self.buffer if item[0] >= start_offset]
        # Snapshot the current buffer
        data = list(self.buffer)
        if data:
            max_offset = max(item[0] for item in data) + 1
        else:
            max_offset = start_offset
        next_read = {"offset": max_offset}
        return iter(data), next_read
      
class GTFSRTDataSource(DataSource):
  @classmethod
  def name(cls):
      """Returns the name of the data source."""
      return "gtfsrt"

  def __init__(self, options):
      """Initialize with options provided."""
      self.options = options

  def schema(self):
      """Returns the schema of the data source."""
      return StructType([
          StructField("offset", IntegerType(), False),
          StructField("url", StringType(), False),
          StructField("binary_proto", BinaryType(), True)
      ])

  # Swap with simple stream reader if using it. Don't use both at the same time.
  # def streamReader(self, schema: StructType):
  #     """Returns an instance of the reader for this data source."""
  #     return GTFSRTStreamReader(schema, self.options)

  def simpleStreamReader(self, schema: StructType):
      """Returns an instance of the reader for this data source."""
      return GTFSRTSimpleStreamReader(schema, self.options)


# Register the source with the spark session
spark.dataSource.register(GTFSRTDataSource)

In [0]:
import requests
from pyspark.sql.datasource import SimpleDataSourceStreamReader, DataSourceStreamReader, DataSource, InputPartition
from pyspark.sql.types import (
    StructType, StructField, BinaryType, StringType, IntegerType
)
from pyspark.sql.functions import current_timestamp
from typing import Iterator, Tuple
import time
from datetime import datetime, timezone

# SIMPLE option
class GTFSRTSimpleStreamReader(SimpleDataSourceStreamReader):
    def __init__(self, schema: StructType, options: dict):
        """Initialize with schema and options."""
        self.schema = schema
        self.url = options["url"]
        self.session = requests.Session()
    
    def initialOffset(self):
        # Initial offset (for a newly created stream) is 0 for the first run.
        return {"offset": 0}

    def read(self, start: dict):
        """
        Takes start offset as an input, then returns an iterator of tuples and the 
        start offset of the next read.
        """
        resp = self.session.get(self.url)
        resp.raise_for_status()
        data = iter([
            (start["offset"], self.url, resp.content),
        ])
        next_read = {"offset": start["offset"] + 1}
        return data, next_read
    


class RequestPartition(InputPartition):
    def __init__(self, i: int, url: str):
        self.i = i
        self.url = url

# More complex option, if controlling partitions is important.
class GTFSRTStreamReader(DataSourceStreamReader):

    def initialOffset(self):
        # Initial offset (for a newly created stream) is 0 for the first run.
        return {"offset": 0}

    def latestOffset(self) -> dict:
        """
        Returns the current latest offset that the next microbatch will read to.
        """
        self.current += 1
        return {"offset": self.current}

    def __init__(self, schema: StructType, options: dict):
        """Initialize with schema and options."""
        super().__init__()
        self.schema = schema
        self.url = options.get("url", "")
        self.current = 0  # Track the offset internally

    def partitions(self, start: dict, end: dict):
        """
        Plans the partitioning of the current microbatch defined by start and end offset. It
        needs to return a sequence of :class:`InputPartition` objects.
        """
        # Just 1 partition since we are only making 1 API call per read.
        return [
            RequestPartition(start["offset"], self.url),
        ]

    def read(self, partition: RequestPartition):
        """
        Reads data starting from the given offset.
        
        Notes
        -----
        This method is static and stateless. You shouldn't access mutable class member
        or keep in memory state between different invocations of read().
        """
        url = partition.url
        resp = requests.get(url)
        resp.raise_for_status()
        yield (partition.i, url, resp.content)
    
    def commit(self, end: dict):
        """
        This is invoked when the query has finished processing data before end offset. This
        can be used to clean up the resource.
        """
        pass

class GTFSRTDataSource(DataSource):
    @classmethod
    def name(cls):
        """Returns the name of the data source."""
        return "gtfsrt"

    def __init__(self, options):
        """Initialize with options provided."""
        self.options = options

    def schema(self):
        """Returns the schema of the data source."""
        return StructType([
            StructField("offset", IntegerType(), False),
            StructField("url", StringType(), False),
            StructField("binary_proto", BinaryType(), True)
        ])

    # Swap with simple stream reader if using it. Don't use both at the same time.
    # def streamReader(self, schema: StructType):
    #     """Returns an instance of the reader for this data source."""
    #     return GTFSRTStreamReader(schema, self.options)

    def simpleStreamReader(self, schema: StructType):
        """Returns an instance of the reader for this data source."""
        return GTFSRTSimpleStreamReader(schema, self.options)


# Register the source with the spark session
spark.dataSource.register(GTFSRTDataSource)

In [0]:
# Use the streaming data source
df = (
      spark.readStream
            .format("gtfsrt")
            .option("url", url)
            .load()
            .withColumn("ingested_at", current_timestamp())
)

In [0]:
# Deserialize protobuf data using native from_protobuf function and the descriptor file
proto_df = df.withColumn(
    "parsed_feed_message", from_protobuf("binary_proto", message_name, descFilePath=descriptor_path).alias("proto_data")
)

In [0]:
# Sink results in to Delta Table
(
  proto_df
    .drop("offset")
    .writeStream
    .outputMode("append")
    .option("checkpointLocation", f"/dbfs/john_armstrong/checkpoints/gtfsrt/{message_type}")
    .option("mergeSchema", "true")
    .trigger(processingTime="5 seconds")
    .toTable("shared.john_armstrong.gtfs_rt_vehicle_positions")
)

In [0]:
dbutils.fs.rm("/dbfs/john_armstrong/checkpoints/gtfsrt", recurse=True)

In [0]:
proto_df = spark.read.table("shared.john_armstrong.gtfs_rt_vehicle_positions_bronze").withColumn(
    "parsed_feed_message", from_protobuf("binary_proto", message_name, descFilePath=descriptor_path)
)